# TPPP look-alikes and prospects — `tppp_prospects_v8.ipynb`

Two tasks, one section each.

| Task | Who | Compared with | Output |
|---|---|---|---|
| **1. Look-alikes** (§2–§5) | PNC TM clients **not registered** as TPPPs | the ~89 registered TPPPs (`vector_customer`), on customer metrics | ranked list: how TPPP-like each client is, the three registered TPPPs it most resembles, and on which metrics |
| **2. Prospects** (§6–§9) | **counterparties** (outside companies, not PNC clients) | registered TPPPs' own accounts at other banks, plus well-known processors, on counterparty metrics | ranked list of the most TPPP-like counterparties, exported for outside enrichment |

**Why the outside accounts of registered TPPPs.** An account that a registered TPPP holds at another bank is a
counterparty, measured with the same counterparty metrics as any prospect and seen the same way — only through its
flows with PNC clients. Its profile is the best available picture of what an outside TPPP looks like from PNC's
side. §7 compares it with the well-known processors; §8 tests which set of examples finds the other.

**Method, both tasks.** Behaviour only (no industry codes or names in the scores). Examples against everyone else,
scored **out of fold** so no example is scored by a model that saw it; the list ranks by the method that finds hidden
examples best. Industry, names and relationship to PNC are shown as columns for the reviewer.

**Status of a hit:** a pattern worth a look, not a finding; prospects are confirmed later with outside sources.
Prospecting use needs the permissible-use review.

**v6 → v7**

| # | Change | Reason |
|---|---|---|
| 1 | folds keep a corporate family (task 1) or all accounts of one company (task 2) together | a hidden Paychex account was scored by a model that had seen eight other Paychex accounts: recall looked better than it is |
| 2 | task 1 reports the stage-1 recall over **all** registered TPPPs as the honest number | reference TPPPs were chosen for scoring high, so their stage-2 recall is flattered |
| 3 | task 1 **archetypes**: reference TPPPs grouped by behaviour (payroll, payments, collections and tax, benefits and claims …); each look-alike carries the archetype it resembles, with a filter | the list mixed insurers, governments and lotteries (like the collection, tax and benefits TPPPs) with payroll look-alikes |
| 4 | brands: no brand from bank-like registered TPPPs (Truist Financial, Sumitomo Mitsui) or `BRAND_EXCLUDE`; known processors sharing a registered TPPP's first word (ADP …) count as its outside accounts | TRUIST LN 001 and SUMITOMO ELECTRI were used as examples; ADP PAY-BY-PAY was a prospect |
| 5 | possible-PNC-client match needs one name's distinctive words (one of ≥ 4 letters) to be all in the other's | EPX ST ↔ EPX FLOORING, ACTUM PROCESSING ↔ ACTUM CONSULTING, ELV BRIGHTWHEE ↔ ELV ALLIANCE were wrong |
| 6 | more generic descriptors (MERCH SERV, MERCH BNKCD NSD) and insurers (UNITED FIN CAS, ARTISAN & TRUCK) excluded | they reached the prospect top 25 |
| 7 | lists show `screened rank` instead of a score that reads 100.0 for the top few hundred | — |

**v7 → v8** (task 1; task 2 unchanged)

| # | Change | Reason |
|---|---|---|
| 1 | who is on each side: outside $ in / out from businesses, PNC $ in / out from organisations | insurers, utilities and governments (people in, people out) looked like payroll processors (businesses in, people out) |
| 2 | archetype = nearest archetype centre, not the archetype of the single most similar TPPP | one unusual TPPP could pull an insurer into the payroll archetype |
| 3 | `LIST_EXCL_NAICS`: insurance carriers, utilities, public administration, fuel wholesale and retail, refineries leave the list (kept in the scores table, `sector_excluded`) | no reference TPPP sits in these sectors and they filled the top 25 |
| 4 | §11 review workbook (Excel): README, look-alikes, prospects, KYC check, archetypes | for the team review after the meeting |

Outputs (`bdahd01p_dlcdi1_cdi_tm`, CSVs in `OUT_DIR`): `pkg_tppp_lookalikes_v8`, `pkg_tppp_prospects_v8`,
`tppp_lookalikes_v8.csv`, `tppp_prospects_v8.csv`, and the de-identified enrichment file.

## §0 Setup

In [ ]:
# §0a  Session and compat shims
import os, re, json, time, math, hashlib, difflib, warnings, datetime as dt, html as _html
warnings.filterwarnings("ignore", category=ResourceWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np
import pandas as pd
with warnings.catch_warnings():                 # hasattr on removed aliases warns in NumPy 2
    warnings.simplefilter("ignore")
    for _alias, _target in [("object0", np.object_), ("bool8", np.bool_), ("int0", np.intp),
                            ("uint0", np.uintp), ("bool", np.bool_), ("object", object)]:   # np.object: non-Arrow toPandas
        if not hasattr(np, _alias):
            setattr(np, _alias, _target)
if not hasattr(pd.DataFrame, "iteritems"):
    pd.DataFrame.iteritems = pd.DataFrame.items
if not hasattr(pd.Series, "iteritems"):
    pd.Series.iteritems = pd.Series.items
import matplotlib
matplotlib.use("Agg")                           # figures go to disk, not inline
import matplotlib.pyplot as plt
from IPython.display import display, HTML
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.utils import AnalysisException
import sklearn
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
try:
    from sklearn.ensemble import HistGradientBoostingClassifier
except ImportError:                              # sklearn < 1.0
    from sklearn.experimental import enable_hist_gradient_boosting  # noqa: F401
    from sklearn.ensemble import HistGradientBoostingClassifier

spark = (SparkSession.builder
         .appName("pkg_tppp_prospects")
         .config("spark.yarn.queue", "root.ua.CIB-AMG.spark")
         .config("spark.executor.memory", "8g")
         .config("spark.executor.cores", "2")
         .config("spark.executor.memoryOverhead", "3g")
         .config("spark.driver.memory", "8g")
         .config("spark.driver.maxResultSize", "4g")
         .config("spark.dynamicAllocation.enabled", "true")
         .config("spark.dynamicAllocation.maxExecutors", "256")
         .config("spark.sql.execution.arrow.pyspark.enabled", "true")
         .config("spark.sql.adaptive.enabled", "true")
         .config("spark.sql.adaptive.skewJoin.enabled", "true")
         .enableHiveSupport()
         .getOrCreate())
spark.conf.set("spark.sql.shuffle.partitions", "800")
spark.conf.set("spark.sql.ansi.enabled", "false")      # ratios with a zero denominator are NULL, not an error
print(f"Spark {spark.version} | pandas {pd.__version__} | numpy {np.__version__} | sklearn {sklearn.__version__}")

In [ ]:
# §0b  Config
DB             = "bdahd01p_dlcdi1_cdi_tm"
CUSTOMER_TABLE = "dsihd01p_dsi.vector_customer"
TPPP_TYPE      = "Third-Party Payment/Payroll Processor"
T_CUST_DIM     = f"{DB}.pkg_cust_dim"
T_CUST_MET     = f"{DB}.pkg_cust_metrics_monthly_v3"
T_CUST_DICT    = f"{DB}.pkg_cust_metrics_dictionary_v3"
T_PAIR         = f"{DB}.pkg_cust_pair_month_v3"
T_CPTY_MET     = f"{DB}.pkg_cpty_metrics_monthly"
T_CPTY_DICT    = f"{DB}.pkg_cpty_metrics_dictionary"
T_ENT_DIM      = f"{DB}.pkg_cpty_entity_dim"

WINDOW_START, WINDOW_END = "2026-01", "2026-08"
STABLE_END        = "2026-06"  # amounts shown as monthly averages up to here (July–August 2026 source jump)
MIN_ACTIVE_MONTHS = 4          # active in ≥ this many window months
MIN_MATCH_RATE    = 0.5        # fail if fewer registered TPPP power ids than this reach pkg_cust_dim

# Task 1 — look-alikes
CUST_PARTY_TYPES = ["O"]
CUST_MIN_AMT     = 50_000      # received AND paid per month
CUST_MIN_CPTY    = 10          # payers or payees per month
REF_MIN_PCT      = 0.90        # a registered TPPP is a reference example when its held-out score is in the top 10%
REF_MIN_N        = 15          # … if fewer qualify, all screened registered TPPPs are used
N_SIMILAR        = 3           # registered TPPPs shown as "most similar"
N_ARCHETYPES     = 5           # behaviour groups of reference TPPPs (k-means on the separating metrics)
ARCHETYPE_SHOW   = None        # e.g. [1, 3]: restrict the look-alike list to these archetypes (None = all)
# sectors left off the look-alike list (prefix match on NAICS); a prefix is kept if any reference TPPP sits in it
LIST_EXCL_NAICS  = ["52411", "52412", "52413", "22", "92", "4247", "4571", "3241"]
REVIEW_N         = 300         # rows per list in the review workbook

# Task 2 — prospects
CPTY_CLASSES       = ["business", "unknown"]
CPTY_MIN_CUST      = 10        # PNC clients per month, either direction (v5: 20; lowered to keep more TPPP outside accounts)
CPTY_MIN_ORG_SHARE = 0.5       # $ share with PNC organisations on at least one side (TPPP prospects are B2B)
BRAND_MAX_FAM      = 20        # a registered TPPP's first name word is a brand if ≤ this many PNC corporate families share it
BRAND_EXCLUDE      = ["TRUIST", "SUMITOMO"]   # registered TPPPs whose first word is not a processor brand
BRAND_BANK_RE      = r"\b(?:BANK|BANKING|BANCORP|BANCSHARES|FINANCIAL CORP|SAVINGS|CREDIT UNION|FSB)\b"
LOOSE_MAX_FREQ     = 5         # a counterparty sharing a distinctive first word with ≤ this many PNC families: possible client
MIN_EX_TPPP        = 10        # choose the prospect model on TPPP outside accounts when at least this many are screened

# Scoring
SCORE_SET           = "shape"  # no $ amounts in the models: the lists are not just the biggest companies
METHODS             = ["gbm", "logit", "knn"]
N_FOLDS, N_REPEATS  = 5, 2
SEED                = 7
GBM_ITER            = 200
MAX_UNLABELED_TRAIN = 150_000
KNN_MIN_SEP         = 0.05
REASON_MIN_SEP      = 0.10
FINAL_METHOD        = "auto"

# Lists and labels (labels never enter the scores)
TOP_N_CUST, TOP_N_CPTY, INTRO_TOP, SHOW_N = 300, 300, 3, 25
PROC_NAICS_SEED     = ["522320", "541214", "522390", "561440"]
PROC_NAICS_MIN_TPPP = 3
PROC_NAME_RE = (r"\b(?:PAY|PAYS|PAYMENT|PAYMENTS|PAYROLL|PROCESSING|PROCESSOR|PROCESSORS|MERCHANT|MERCHANTS|SETTLEMENT|"
                r"CLEARING|FBO|ESCROW|DISBURSEMENT|DISBURSEMENTS|REMITTANCE|BILLING|COLLECTION|COLLECTIONS)\b")
BANK_NAME_RE = r"\b(?:BANK|BANCORP|BANCSHARES|BANCORPORATION|CREDIT UNION|SAVINGS|FSB)\b"
PNC_NAME_RE  = r"\bPNC\b"
EXCL_NAICS_PREFIX = ["521", "5221", "92"]
UNLIKELY_NAICS2 = []           # v6: no industry filter on the look-alikes (shown as a column instead)
TPPP_SECTOR_MIN = 2
KNOWN_PROC_RE = (r"\b(?:ADP|PAYCHEX|PAYCOM|PAYLOCITY|DAYFORCE|CERIDIAN|GUSTO|PAYCOR|ISOLVED|TRINET|JUSTWORKS|INSPERITY|"
                 r"RIPPLING|STRIPE|PAYPAL|BRAINTREE|WORLDPAY|FISERV|FIRST DATA|GLOBAL PAYMENTS|HEARTLAND PAYMENT|TSYS|"
                 r"ELAVON|PAYMENTECH|AUTHORIZE NET|SHIFT4|NUVEI|ADYEN|PAYSAFE|REPAY|DWOLLA|BILL COM|AVIDXCHANGE|"
                 r"CLICKPAY|APPFOLIO|YARDI|PAYLEASE|ZEGO|REALPAGE)\b")
CPTY_EXCL_RE = (r"\b(?:VISA|MASTERCARD|AMERICAN EXPRESS|AMEX|DISCOVER|IRS|TREASURY|TREAS|SSA|STATE OF|DEPT OF|"
                r"DEPARTMENT OF|COUNTY|CITY OF|COMMONWEALTH|INSURANCE|ASSURANCE|MUTUAL|FANNIE MAE|FREDDIE MAC|"
                r"FEDERAL HOME LOAN|CITICARD|CITI CARDS?|COMENITY|SYNCHRONY|CAPITAL ONE|CHASE CARD|CARD SERVICES|"
                r"HONDA|HYUNDAI|TOYOTA|NISSAN|KIA MOTORS|FORD CREDIT|GM FINANCIAL|STATE FARM|GEICO|PROGRESSIVE|"
                r"ALLSTATE|NATIONWIDE|LIBERTY MUTUAL|USAA)\b"
                r"|SALESTAX|SALES TAX|USETAX|USE TAX|TAXPYMT|DEPT REV|\bCAS\b|CASUALTY|ARTISAN TRUCK|\bINS\b|INS CO\b")
GENERIC_TOKENS = {"BNKCD", "BANKCARD", "BANKCD", "BNKCRD", "MERCH", "MERCHANT", "MERCHANTS", "MRCH", "SVC", "SVCS",
                  "SERVICE", "SERVICES", "SETTLE", "SETTLEMENT", "STLMT", "DEP", "DEPOSIT", "EMPLOYEE", "BENEFIT",
                  "BENEFITS", "PAYMENT", "PAYMENTS", "PAY", "CARD", "CARDS", "DEBIT", "CREDIT", "ACH", "TRANSFER",
                  "FEE", "FEES", "DISC", "DISCOUNT", "BATCH", "PAYROLL", "REFUND", "PROCESSING", "DEPOSITS", "CHGBK",
                  "MER", "BK", "BNK", "MRCHNT", "MERCHNT", "BANKCRD", "STLMNT", "SETL", "CRD", "DEPST",
                  "SERV", "SERVS", "SRV", "SRVS", "NSD", "SD", "STL", "STLM", "CHGBCK", "DISCNT", "ACCT", "ACC"}
FIRST_WORD_STOP = ["THE", "AMERICAN", "AMERICA", "FIRST", "NATIONAL", "UNITED", "GLOBAL", "GENERAL", "INTERNATIONAL",
                   "CITY", "COUNTY", "STATE", "NORTH", "SOUTH", "EAST", "WEST", "CENTRAL", "PACIFIC", "ATLANTIC",
                   "PREMIER", "ADVANCED", "ALLIED", "STANDARD", "SUPERIOR", "QUALITY", "NEW", "US", "USA", "ALL",
                   "PAY", "PAYROLL", "PAYMENT", "PAYMENTS", "MERCHANT", "SERVICES", "SERVICE", "GROUP", "HOLDINGS"]

OUT_DIR          = "../metrics/tppp/v8"
WRITE_TABLES     = True
REBUILD_EXTRACTS = False
VERSION          = "tppp_v8"
os.makedirs(OUT_DIR, exist_ok=True)
assert SCORE_SET in ("shape", "full") and set(METHODS) <= {"gbm", "logit", "knn"} and METHODS
print(f"window {WINDOW_START} → {WINDOW_END} (amounts to {STABLE_END}) | methods {METHODS} | outputs {OUT_DIR}")

In [ ]:
# §0c  Helpers: months, display, ids and names, feature resolution, scoring
def month_add(m, k):
    t = int(m[:4]) * 12 + int(m[5:7]) - 1 + k
    return f"{t // 12:04d}-{t % 12 + 1:02d}"

def month_range(a, b):
    out, m = [], a
    while m <= b:
        out.append(m)
        m = month_add(m, 1)
    return out

WINDOW = month_range(WINDOW_START, WINDOW_END)

def table_exists(t):
    try:
        spark.table(t).columns
        return True
    except AnalysisException:
        return False

def partitions(table):
    out = set()
    for r in spark.sql(f"SHOW PARTITIONS {table}").collect():
        for kv in r[0].split("/"):
            if kv.startswith("month="):
                out.add(kv.split("=", 1)[1])
    return out

def db_location(db):
    for r in spark.sql(f"DESCRIBE DATABASE EXTENDED {db}").collect():
        if str(r[0]).strip().lower() == "location":
            return str(r[1]).rstrip("/")
    raise RuntimeError(f"no Location row in DESCRIBE DATABASE {db}")

_jvm, _hconf = spark._jvm, spark._jsc.hadoopConfiguration()
def _fs(p):
    path = _jvm.org.apache.hadoop.fs.Path(p)
    return path.getFileSystem(_hconf), path
def exists(p):
    fs, path = _fs(p)
    return fs.exists(path)
def touch(p):
    fs, path = _fs(p)
    fs.create(path, True).close()

LOC = db_location(DB)
P_TMP = f"{LOC}/_tmp_pkg_tppp"

# ---- display: one output per cell ----
def _fmt(v, name):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return ""
    if isinstance(v, (bool, np.bool_)):
        return "✓" if v else ""
    if isinstance(v, str):
        return v
    if not isinstance(v, (int, float, np.integer, np.floating)):
        return str(v)
    v = float(v)
    if name.startswith("%") or name.endswith("%") or name.startswith("median %"):
        return f"{v:.1f}"
    if "$" in name:
        return f"{v:,.0f}" if abs(v) >= 10 else f"{v:,.1f}"
    if v.is_integer():
        return f"{v:,.0f}"
    return f"{v:,.2f}" if abs(v) >= 1 else f"{v:.3f}"

def fmt_df(df):
    out = df.copy()
    out.columns = [str(c) for c in out.columns]
    for c in out.columns:
        out[c] = [_fmt(v, c) for v in out[c].tolist()]
    return out

_CSS = ("<style>.pkgt{border-collapse:collapse;font-size:12px;margin-bottom:4px}"
        ".pkgt th,.pkgt td{padding:2px 8px;border-bottom:1px solid #e3e2de;text-align:right;white-space:nowrap}"
        ".pkgt td:nth-child(-n+3){text-align:left}"
        ".pkgt th{background:#e3e2de;color:#0b0b0b !important;font-weight:600}.pkgn{color:#6f6e69;margin:2px 0}</style>")

def note(text):
    return ("__note__", text)

def render(*blocks):
    parts = []
    for b in blocks:
        if b is None:
            continue
        if isinstance(b, str):
            parts.append(f"<div style='margin:10px 0 2px;font-size:14px'><b>{_html.escape(b)}</b></div>")
        elif b[0] == "__note__":
            parts.append(f"<div class='pkgn'>{_html.escape(b[1])}</div>")
        elif b[1] is not None:
            parts.append(f"<div style='margin:12px 0 3px'><b>{_html.escape(b[0])}</b></div>")
            parts.append(fmt_df(b[1]).to_html(index=False, escape=True, border=0, classes="pkgt"))
    # tex2jax_ignore / mathjax_ignore: JupyterLab must not read text between two dollar signs as math
    return HTML(_CSS + "<div class='tex2jax_ignore mathjax_ignore' style='font-family:\"IBM Plex Sans\",Arial,sans-serif'>"
                + "".join(parts) + "</div>")

def show(*blocks):
    display(render(*blocks))

# ---- ids and names ----
def norm_pwr(c):
    # power id as a 10-digit string with leading zeros (matches tppp_scatter_explorer_v2); '123.0' → '0000000123'
    s = F.regexp_replace(F.trim(c.cast("string")), r"\.0+$", "")
    return F.when(F.length(s) > 0, F.lpad(s, 10, "0"))

LEGAL = ["INC", "INCORPORATED", "LLC", "LLP", "LP", "LTD", "LIMITED", "CORP", "CORPORATION", "CO", "COMPANY", "PC",
         "PLLC", "PA", "PLC", "NA", "THE", "DBA"]
def name_norm(c):
    return F.trim(F.regexp_replace(F.upper(F.coalesce(c, F.lit(""))), r"[^A-Z0-9]+", " "))

def name_key(c):
    # same rule as pkg_cpty_entity_dim.pnc_match_n: legal words removed, single letters joined (L L C → LLC)
    k = F.regexp_replace(F.upper(c), r"[^A-Z0-9]+", " ")
    k = F.regexp_replace(F.trim(k), r"\b([A-Z]) (?=[A-Z]\b)", "$1")
    k = F.regexp_replace(k, r"\b(" + "|".join(sorted(LEGAL, key=len, reverse=True)) + r")\b", " ")
    k = F.trim(F.regexp_replace(k, r"\s+", " "))
    return F.when(F.length(k) >= 5, k)

# ---- feature specs ----
def feat(name, cands, label, kind="shape", fmt="num", model=True):
    return dict(name=name, cands=cands, label=label, kind=kind, fmt=fmt, model=model)

def both(name, base, label_in, label_out, kind="shape", fmt="num", model=True, upper=False):
    # one feature per direction; candidates cover lower / upper-case column spellings
    out = []
    for d, lab in (("in", label_in), ("out", label_out)):
        cands = [f"{b}_{d}" for b in (base if isinstance(base, list) else [base])]
        out.append(feat(f"{name}_{d}", cands, lab, kind, fmt, model))
    return out

def resolve(spec, columns, dict_df=None):
    low = {c.lower(): c for c in columns}
    rows, got = [], {}
    for f in spec:
        hit = next((low[c.lower()] for c in f["cands"] if c.lower() in low), None)
        if hit:
            got[f["name"]] = hit
        close = "" if hit else ", ".join(difflib.get_close_matches(f["cands"][0].lower(), list(low), n=3, cutoff=0.6))
        definition = ""
        if hit is not None and dict_df is not None:
            m = dict_df[dict_df.column.str.lower() == hit.lower()]
            definition = str(m.definition.iloc[0]) if len(m) else ""
        rows.append({"feature": f["name"], "column": hit or "MISSING", "in model": f["model"], "kind": f["kind"],
                     "label": f["label"], "closest columns (if missing)": close, "dictionary definition": definition})
    return got, pd.DataFrame(rows)

def read_dict(t):
    if not table_exists(t):
        return None
    d = spark.table(t).toPandas()
    return d if {"column", "definition"} <= set(d.columns) else None

def fmt_value(v, kind):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return "n/a"
    if kind == "usd":       # no dollar sign: cells with two of them render as math in JupyterLab
        return f"{v / 1e6:,.1f}M" if abs(v) >= 1e6 else f"{v / 1e3:,.0f}K" if abs(v) >= 1e3 else f"{v:,.0f}"
    if kind == "pct":
        return f"{100 * v:.0f}%"
    if kind == "ratio":
        return f"{v:.2f}"
    if kind == "int":
        return f"{v:,.0f}"
    return f"{v:,.1f}"

# ---- scoring (shared by look-alikes and prospects) ----
def separation(df, feats, y):
    # single-feature AUC of examples vs the rest, on rows where the feature exists
    rows = []
    for f in feats:
        v = pd.to_numeric(df[f], errors="coerce")
        ok = v.notna().to_numpy()
        auc = np.nan
        if ok[y].sum() >= 3 and ok[~y].sum() >= 3 and v[ok].nunique() > 1:
            auc = roc_auc_score(y[ok], v[ok])
        rows.append({"feature": f, "AUC": auc, "median examples": v[y].median(), "median others": v[~y].median(),
                     "% examples with value": 100 * ok[y].mean(), "% others with value": 100 * ok[~y].mean()})
    s = pd.DataFrame(rows)
    s["separation"] = (s.AUC - 0.5).abs()
    return s.sort_values("separation", ascending=False).reset_index(drop=True)

def rank_matrix(df, feats):
    # percentile rank within the screened population; missing → 0.5 (neutral)
    return df[feats].apply(pd.to_numeric, errors="coerce").rank(pct=True).fillna(0.5).to_numpy(float)

def _fit_predict(method, Xtr, ytr, wtr, Xte, seed):
    if method == "gbm":
        m = HistGradientBoostingClassifier(max_iter=GBM_ITER, learning_rate=0.05, max_leaf_nodes=15,
                                           min_samples_leaf=20, l2_regularization=1.0, random_state=seed)
    else:
        m = LogisticRegression(C=0.5, max_iter=1000)
    m.fit(Xtr, ytr, sample_weight=wtr)
    return m.predict_proba(Xte)[:, 1]

def cv_pct(method, X, y, train_ok, seed0=SEED):
    # Out-of-fold percentile (within the held-out fold), averaged over repeats. Rows with train_ok = False are
    # scored but never trained on (e.g. family members of registered TPPPs, whose status is unknown).
    n, rng = len(y), np.random.default_rng(seed0)
    acc = np.zeros(n)
    for r in range(N_REPEATS):
        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed0 + r)
        pct = np.full(n, np.nan)
        for tr, te in skf.split(np.zeros(n), y):
            tr = tr[train_ok[tr]]
            pos, neg = tr[y[tr]], tr[~y[tr]]
            if len(neg) > MAX_UNLABELED_TRAIN:
                neg = rng.choice(neg, MAX_UNLABELED_TRAIN, replace=False)
            tr = np.concatenate([pos, neg])
            w = np.where(y[tr], len(neg) / max(len(pos), 1), 1.0)       # balance the classes
            s = _fit_predict(method, X[tr], y[tr], w, X[te], seed0 + r)
            pct[te] = pd.Series(s).rank(pct=True).to_numpy()
        acc += pct
    return acc / N_REPEATS

def knn_pct(R, y, w, chunk=50_000):
    # Weighted distance (percentile space) to the nearest example; an example's own row is skipped (leave-one-out).
    pos = np.where(y)[0]
    sw = np.sqrt(w / w.sum())
    P = R[pos] * sw
    pn = (P ** 2).sum(1)
    best, dmin = np.zeros(len(y), int), np.zeros(len(y))
    for a in range(0, len(y), chunk):
        X = R[a:a + chunk] * sw
        d2 = (X ** 2).sum(1)[:, None] + pn[None, :] - 2 * X @ P.T
        rows = np.where((pos >= a) & (pos < a + chunk))[0]
        d2[pos[rows] - a, rows] = np.inf
        best[a:a + chunk] = d2.argmin(1)
        dmin[a:a + chunk] = np.sqrt(np.maximum(d2.min(1), 0))
    return pd.Series(-dmin).rank(pct=True).to_numpy(), pos[best], dmin

def score_population(df, feats, y, train_ok, sep, methods=METHODS):
    # returns {method: percentile}, nearest-example row index, nearest distance
    out, near, dist = {}, None, None
    R = rank_matrix(df, feats)
    X = df[feats].apply(pd.to_numeric, errors="coerce").to_numpy(float)
    for m in methods:
        if m == "gbm":
            out[m] = cv_pct("gbm", X, y, train_ok)
        elif m == "logit":
            out[m] = cv_pct("logit", R, y, train_ok)
    # the nearest example is always computed: it is the explanation shown on every list
    s = sep.set_index("feature").reindex(feats)
    w = np.clip(s.separation.fillna(0).to_numpy() - KNN_MIN_SEP, 0, None)
    w = w if w.sum() > 0 else np.ones(len(feats))
    kp, near, dist = knn_pct(R, y, w)
    if "knn" in methods:
        out["knn"] = kp
    out["blend"] = np.nanmean(np.vstack([out[m] for m in methods]), axis=0)
    return out, near, dist

KS = [0.001, 0.005, 0.01, 0.05]
def recall_table(pcts, y, groups=None):
    # held-out rank of the examples: share of the screened population ranked above each one
    rows, N = [], len(y)
    for name, p in pcts.items():
        for g, mask in (groups or {"all examples": y}).items():
            q = 1 - p[mask]
            if len(q) == 0:
                continue
            r = {"method": name, "examples": g, "n": int(mask.sum()), "median % ranked above": 100 * np.median(q)}
            for k in KS:
                r[f"found in top {100 * k:g}% ({int(round(k * N)):,} rows)"] = int((q <= k).sum())
            rows.append(r)
    return pd.DataFrame(rows)

def pick_method(rec, pcts, group="all examples"):
    # FINAL_METHOD, or the scoring method with the lowest held-out median rank (ties → first in METHODS + blend)
    if FINAL_METHOD != "auto":
        assert FINAL_METHOD in pcts, f"FINAL_METHOD {FINAL_METHOD} not scored; have {list(pcts)}"
        return FINAL_METHOD
    r = rec[rec.method.isin(list(pcts)) & (rec.examples == group)]
    return r.sort_values("median % ranked above").method.iloc[0]

def recall_figure(pcts, y, path, title):
    xs = np.logspace(-4, 0, 200)
    fig, ax = plt.subplots(figsize=(6.5, 4.2))
    for name, p in pcts.items():
        q = np.sort(1 - p[y])
        ax.plot(xs, [(q <= x).mean() for x in xs], label=name, lw=2 if name == "blend" else 1.2)
    ax.plot(xs, xs, ls=":", c="grey", label="random")
    ax.set_xscale("log")
    ax.set_xlabel("share of screened population reviewed (top-ranked first)")
    ax.set_ylabel("share of held-out examples found")
    ax.set_title(title, fontsize=10)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(path, dpi=130)
    plt.close(fig)

def reasons(df, feats, sep, spec_by_name, top=3):
    # Reason codes: the features on which a row is most example-like, as oriented percentiles among the screened
    s = sep.set_index("feature").reindex(feats)
    use = [f for f in feats if s.loc[f, "separation"] >= REASON_MIN_SEP]
    if not use:
        return pd.Series([""] * len(df), index=df.index)
    V = df[use].apply(pd.to_numeric, errors="coerce")
    R = V.rank(pct=True)
    O = np.where((s.loc[use, "AUC"] > 0.5).to_numpy()[None, :], R.to_numpy(), 1 - R.to_numpy())
    O = np.nan_to_num(O, nan=0.0)
    idx = np.argsort(-O, axis=1)[:, :top]
    out = []
    for i in range(len(df)):
        parts = []
        for j in idx[i]:
            f = use[j]
            if O[i, j] < 0.75:
                continue
            sp = spec_by_name[f]
            top_ = 100 * (1 - O[i, j])
            parts.append(f"{sp['label'].replace('$ ', '')} {fmt_value(V.iat[i, j], sp['fmt'])} "
                         f"(top {'<0.1' if top_ < 0.1 else f'{top_:.1f}'}%)")
        out.append("; ".join(parts))
    return pd.Series(out, index=df.index)

def to_sdf(pdf):
    # explicit schema; NaN → NULL in text columns
    typ = {"object": "string", "bool": "boolean", "int64": "bigint", "int32": "int", "float64": "double",
           "Int64": "bigint", "boolean": "boolean"}
    pdf = pdf.copy()
    for c in pdf.columns:
        if pdf[c].dtype == object:
            pdf[c] = pdf[c].astype(object).where(pdf[c].notna(), None)
    schema = ", ".join(f"`{c}` {typ.get(str(pdf[c].dtype), 'string')}" for c in pdf.columns)
    return spark.createDataFrame(pdf, schema)

def write_table(pdf, name):
    path = f"{LOC}/{name}"
    to_sdf(pdf).repartition(4).write.mode("overwrite").parquet(path)
    spark.sql(f"DROP TABLE IF EXISTS {DB}.{name}")
    cols = ", ".join(f"`{f.name}` {f.dataType.simpleString()}" for f in spark.read.parquet(path).schema.fields)
    spark.sql(f"CREATE EXTERNAL TABLE {DB}.{name} ({cols}) STORED AS PARQUET LOCATION '{path}'")
    return path

def extract(path, sig, build):
    # write once, read back; rebuilt when the signature changes or REBUILD_EXTRACTS is set
    mark = f"{path}/_pkg_sig_{sig}"
    rebuilt = REBUILD_EXTRACTS or not (exists(path + "/_SUCCESS") and exists(mark))
    if rebuilt:
        build().write.mode("overwrite").parquet(path)
        touch(mark)
    return spark.read.parquet(path), rebuilt

print(f"window months: {len(WINDOW)} | scratch {P_TMP}")

# ---- v7: folds that keep groups together (a family, or one company's accounts) ----
try:
    from sklearn.model_selection import StratifiedGroupKFold
except ImportError:                              # sklearn < 1.0
    StratifiedGroupKFold = None
CV_GROUPS = None                                 # set per task before scoring; None = ordinary stratified folds

def cv_pct(method, X, y, train_ok, seed0=SEED):
    n, rng = len(y), np.random.default_rng(seed0)
    acc = np.zeros(n)
    for r in range(N_REPEATS):
        if CV_GROUPS is not None and StratifiedGroupKFold is not None:
            splits = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed0 + r).split(
                np.zeros(n), y, CV_GROUPS)
        else:
            splits = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed0 + r).split(np.zeros(n), y)
        pct = np.full(n, np.nan)
        for tr, te in splits:
            tr = tr[train_ok[tr]]
            pos, neg = tr[y[tr]], tr[~y[tr]]
            if len(neg) > MAX_UNLABELED_TRAIN:
                neg = rng.choice(neg, MAX_UNLABELED_TRAIN, replace=False)
            tr = np.concatenate([pos, neg])
            w = np.where(y[tr], len(neg) / max(len(pos), 1), 1.0)
            s = _fit_predict(method, X[tr], y[tr], w, X[te], seed0 + r)
            pct[te] = pd.Series(s).rank(pct=True).to_numpy()
        acc += pct
    return acc / N_REPEATS

# ---- v6 additions ----
STABLE = [m for m in WINDOW if m <= STABLE_END]
stop_sql = ", ".join("'" + w + "'" for w in FIRST_WORD_STOP)
def first_word(col):
    return F.expr(f"element_at(filter(split({col}, ' '), t -> length(t) >= 3 AND NOT array_contains(array({stop_sql}), t)), 1)")

def knn_top(R, pos, w, k, chunk=20_000):
    # the k nearest examples (rows `pos`) to every row, in weighted percentile space; an example skips itself
    sw = np.sqrt(w / w.sum())
    Pm = R[pos] * sw
    pn = (Pm ** 2).sum(1)
    k = min(k, len(pos) - 1) if len(pos) > 1 else 1
    idx, dst = np.zeros((len(R), k), int), np.zeros((len(R), k))
    for a in range(0, len(R), chunk):
        X = R[a:a + chunk] * sw
        d2 = (X ** 2).sum(1)[:, None] + pn[None, :] - 2 * X @ Pm.T
        rows = np.where((pos >= a) & (pos < a + chunk))[0]
        d2[pos[rows] - a, rows] = np.inf
        o = np.argsort(d2, axis=1)[:, :k]
        idx[a:a + chunk] = pos[o]
        dst[a:a + chunk] = np.sqrt(np.maximum(np.take_along_axis(d2, o, 1), 0))
    return idx, dst

def knn_weights(sep, feats):
    s = sep.set_index("feature").reindex(feats)
    w = np.clip(s.separation.fillna(0).to_numpy() - KNN_MIN_SEP, 0, None)
    return w if w.sum() > 0 else np.ones(len(feats))

def shared_prep(df, feats, sep):
    s = sep.set_index("feature").reindex(feats)
    use = [f for f in feats if s.loc[f, "separation"] >= REASON_MIN_SEP]
    V = df[use].apply(pd.to_numeric, errors="coerce")
    Rk = V.rank(pct=True)
    O = Rk.copy()
    for f in use:
        if s.loc[f, "AUC"] <= 0.5:
            O[f] = 1 - Rk[f]
    return use, V.to_numpy(float), Rk.to_numpy(float), O.to_numpy(float)

def shared_metrics(prep, spec, i, j, top=3):
    # metrics on which row i and example j are both on the example side (top quarter, oriented) and closest together
    use, V, Rk, O = prep
    out = []
    for k, f in enumerate(use):
        oi, oj = O[i, k], O[j, k]
        if np.isfinite(oi) and np.isfinite(oj) and oi >= 0.75 and oj >= 0.75:
            out.append((abs(Rk[i, k] - Rk[j, k]), f, V[i, k], V[j, k]))
    out.sort()
    return "; ".join(f"{spec[f]['label'].replace('$ ', '')} {fmt_value(a, spec[f]['fmt'])} (vs {fmt_value(b, spec[f]['fmt'])})"
                     for _, f, a, b in out[:top])

## §1 Registered TPPPs

Latest `load_date` snapshot of `vector_customer`; power ids normalised to 10 digits on both sides of the join to
`pkg_cust_dim`. Family members (same `master_pwr_id`) are kept apart: neither examples nor look-alikes.

In [ ]:
# §1  Registered TPPP power ids → mdm_ids
tppp_ids = (
    spark.sql(f'''
        select cust_pwr_id
        from {CUSTOMER_TABLE}
        where customer_type = '{TPPP_TYPE}'
          and load_date = (select max(load_date) from {CUSTOMER_TABLE})
    ''')
    .select(norm_pwr(F.col("cust_pwr_id")).alias("customer_pwr_id"))
    .where(F.col("customer_pwr_id").isNotNull())
    .distinct()
    .withColumn("is_tppp", F.lit(True))
    .cache()
)
n_tppp_ids = tppp_ids.count()

CDIM = (spark.table(T_CUST_DIM)
        .select("mdm_id", norm_pwr(F.col("customer_pwr_id")).alias("customer_pwr_id"),
                norm_pwr(F.col("master_pwr_id")).alias("master_pwr_id"), "customer_name", "party_type",
                "naics_cd", "naics_desc", "naics2", "naics_status",
                F.coalesce(F.col("is_tm").cast("boolean"), F.lit(False)).alias("is_tm"),
                F.coalesce(F.col("pwr_shared").cast("boolean"), F.lit(False)).alias("pwr_shared")))
tppp_m = tppp_ids.join(CDIM, "customer_pwr_id", "left").cache()
TPPP = tppp_m.where(F.col("mdm_id").isNotNull()).toPandas()
n_matched = TPPP.customer_pwr_id.nunique()
assert n_tppp_ids > 0, f"no rows with customer_type = '{TPPP_TYPE}' in the latest snapshot of {CUSTOMER_TABLE}"
assert n_matched >= MIN_MATCH_RATE * n_tppp_ids, (
    f"only {n_matched} of {n_tppp_ids} TPPP power ids found in {T_CUST_DIM}: check the id format on both sides")
unmatched = tppp_m.where(F.col("mdm_id").isNull()).select("customer_pwr_id").toPandas()

fam_ids = set(TPPP.master_pwr_id.dropna())
FAMILY = (CDIM.where(F.col("master_pwr_id").isin(list(fam_ids)) if fam_ids else F.lit(False))
              .select("mdm_id").toPandas())
FAMILY_IDS = set(FAMILY.mdm_id) - set(TPPP.mdm_id)
TPPP_IDS = set(TPPP.mdm_id)

TPPP["naics6"] = np.where(TPPP.naics_status == "valid", TPPP.naics_cd.astype(str).str[:6], None)
nc = (TPPP.dropna(subset=["naics6"]).groupby(["naics6"]).agg(n=("mdm_id", "nunique"), desc=("naics_desc", "first"))
          .reset_index().sort_values("n", ascending=False))
PROC_NAICS = sorted(set(PROC_NAICS_SEED) | set(nc.loc[nc.n >= PROC_NAICS_MIN_TPPP, "naics6"]))
counts = pd.DataFrame([
    {"item": "TPPP power ids in latest snapshot", "n": n_tppp_ids},
    {"item": "found in pkg_cust_dim", "n": n_matched},
    {"item": "mdm_ids (a power id can hold several)", "n": len(TPPP_IDS)},
    {"item": "power ids shared by 2+ mdm_ids (pwr_shared)", "n": int(TPPP.loc[TPPP.pwr_shared, "customer_pwr_id"].nunique())},
    {"item": "TM customers", "n": int(TPPP.is_tm.sum())},
    {"item": "family members kept apart (same master_pwr_id)", "n": len(FAMILY_IDS)},
])
pt = TPPP.party_type.fillna("(null)").value_counts().rename_axis("party_type").reset_index(name="mdm_ids")
show(f"§1 Registered TPPPs: {n_matched} of {n_tppp_ids} power ids matched",
     ("Counts", counts),
     ("Party type", pt),
     ("NAICS of registered TPPPs (valid codes)", nc.rename(columns={"naics6": "naics", "n": "TPPPs"}).head(15)),
     note(f"Processor-NAICS evidence set: {', '.join(PROC_NAICS)} (seed + codes held by ≥ {PROC_NAICS_MIN_TPPP} TPPPs)."),
     ("Unmatched power ids (first 20)", unmatched.head(20)) if len(unmatched) else None)

# Task 1 — Look-alikes: PNC clients behaving like registered TPPPs

## §2 Features and screen

Customer metrics only (`pkg_cust_metrics_monthly_v3`), monthly averages over the window. The screen keeps TM
organisations big and busy enough to be a processor; the funnel shows how many registered TPPPs it keeps.

In [ ]:
# §2a  Customer feature spec → table columns
CUST_SPEC = [
    feat("amt_in", ["amt_all_in"], "$ received /mo", "size", "usd"),
    feat("amt_out", ["amt_all_out"], "$ paid /mo", "size", "usd"),
    feat("passthrough", ["passthrough_ratio"], "in/out balance", fmt="ratio"),
    *both("ncpty", "ncpty_all", "payers /mo", "payees /mo", fmt="int"),
    *both("top1", "top1_share", "largest payer share", "largest payee share", fmt="pct"),
    *both("person", "ext_person_share", "external $ in from persons", "external $ out to persons", fmt="pct"),
    *both("ext", "ext_share", "$ in from outside PNC", "$ out to outside PNC", fmt="pct"),
    *both("naics_ent", "naics2_entropy", "industry spread of PNC payers", "industry spread of PNC payees"),
    *both("days", "active_days", "days with money in /mo", "days with money out /mo"),
    *both("ticket", "avg_ticket", "average payment in", "average payment out", fmt="usd"),
    *both("ach", ["share_ACH", "share_ach"], "ACH share in", "ACH share out", fmt="pct"),
    *both("wire", ["share_WIRE", "share_wire"], "wire share in", "wire share out", fmt="pct"),
    *both("rails", "n_rails", "rails in", "rails out"),
    *both("nfi", "n_fi", "banks of external payers", "banks of external payees", fmt="int"),
    *both("retention", "retention", "payer retention", "payee retention", fmt="pct"),
    *both("rec", "rec_standing", "recurring payers", "recurring payees", fmt="int"),
    *both("new_amt", "new_amt_share", "$ in from new payers", "$ out to new payees", fmt="pct"),
    *both("selfpay", "selfpay_share", "$ in from own accounts elsewhere", "$ out to own accounts elsewhere", fmt="pct"),
    *both("biz", "ext_business_share", "outside $ in from businesses", "outside $ out to businesses", fmt="pct"),
    *both("intorg", "int_org_share", "PNC $ in from organisations", "PNC $ out to organisations", fmt="pct"),
    # not in the models: used in §9
    *both("amt_int", "amt_int", "$ in from PNC customers", "$ out to PNC customers", "size", "usd", model=False),
    *both("amt_ext", "amt_ext", "$ in from outside PNC", "$ out to outside PNC", "size", "usd", model=False),
]
CUST_CORE = ["amt_in", "amt_out", "ncpty_in", "ncpty_out"]
CUST_BY_NAME = {f["name"]: f for f in CUST_SPEC}
CUST_DICT = read_dict(T_CUST_DICT)
CUST_COL, cres = resolve(CUST_SPEC, spark.table(T_CUST_MET).columns, CUST_DICT)
miss_core = [f for f in CUST_CORE if f not in CUST_COL]
assert not miss_core, f"core customer features missing: {miss_core} — fix CUST_SPEC (see the table)"
CUST_FEATS = [f["name"] for f in CUST_SPEC if f["model"] and f["name"] in CUST_COL]
CUST_MODEL_FEATS = [f for f in CUST_FEATS if SCORE_SET == "full" or CUST_BY_NAME[f]["kind"] != "size"]
have = partitions(T_CUST_MET)
miss_m = [m for m in WINDOW if m not in have]
assert not miss_m, f"{T_CUST_MET}: window months missing {miss_m}"
n_miss = int((cres.column == "MISSING").sum())
show(f"§2a Customer features: {len(CUST_COL)} of {len(CUST_SPEC)} resolved, {len(CUST_MODEL_FEATS)} used by the models",
     ("Resolution", cres.sort_values(["column"], key=lambda s: s != "MISSING", ascending=True) if n_miss else cres),
     note(f"Score set '{SCORE_SET}': " + ("$ amounts are left out of the models (size alone does not make a processor)."
                                          if SCORE_SET == "shape" else "all features in the models.")),
     note("No dictionary table found: definitions empty.") if CUST_DICT is None else None)

In [ ]:
# §2b  Window averages per TM organisation (written once), screen, funnel
cols_used = sorted(set(CUST_COL.values()))
SIG_C = hashlib.md5(json.dumps({"v": VERSION, "w": WINDOW, "s": STABLE_END, "c": cols_used,
                                "p": CUST_PARTY_TYPES}).encode()).hexdigest()[:10]

def build_cust_agg():
    pop = CDIM.where(F.col("is_tm") & F.col("party_type").isin(CUST_PARTY_TYPES)).select("mdm_id")
    m = (spark.table(T_CUST_MET).where(F.col("month").isin(WINDOW))
              .select("mdm_id", "month", *[F.col(c).cast("double").alias(c) for c in cols_used])
              .join(pop, "mdm_id", "left_semi"))
    act = (F.coalesce(F.col(CUST_COL["amt_in"]), F.lit(0.0)) + F.coalesce(F.col(CUST_COL["amt_out"]), F.lit(0.0))) > 0
    aggs = [F.sum(act.cast("int")).alias("months_active")]
    aggs += [F.avg(F.when(act, F.col(CUST_COL[f]))).alias(f) for f in CUST_COL]
    st = act & F.col("month").isin(STABLE)
    aggs += [F.avg(F.when(st, F.col(CUST_COL["amt_in"]))).alias("amt_in_st"),
             F.avg(F.when(st, F.col(CUST_COL["amt_out"]))).alias("amt_out_st")]
    return m.groupBy("mdm_id").agg(*aggs).where(F.col("months_active") > 0)

cagg, rebuilt_c = extract(f"{P_TMP}/cust_agg", SIG_C, build_cust_agg)
lab = spark.createDataFrame(pd.DataFrame({"mdm_id": sorted(TPPP_IDS | FAMILY_IDS)}).assign(
          is_tppp=lambda d: d.mdm_id.isin(TPPP_IDS), tppp_family=lambda d: ~d.mdm_id.isin(TPPP_IDS)))
cagg_l = cagg.join(F.broadcast(lab), "mdm_id", "left")
cagg_l = (cagg_l.withColumn("is_tppp", F.coalesce("is_tppp", F.lit(False)))
                .withColumn("tppp_family", F.coalesce("tppp_family", F.lit(False))))
s1 = F.col("months_active") >= MIN_ACTIVE_MONTHS
s2 = s1 & (F.col("amt_in") >= CUST_MIN_AMT) & (F.col("amt_out") >= CUST_MIN_AMT)
s3 = s2 & ((F.col("ncpty_in") >= CUST_MIN_CPTY) | (F.col("ncpty_out") >= CUST_MIN_CPTY))
fun = cagg_l.agg(*[F.sum(c.cast("int")).alias(f"n{i}") for i, c in enumerate([F.lit(True), s1, s2, s3])],
                 *[F.sum((c & F.col("is_tppp")).cast("int")).alias(f"t{i}") for i, c in enumerate([F.lit(True), s1, s2, s3])]
                 ).first()
tppp_tm_org = int((TPPP.is_tm & TPPP.party_type.isin(CUST_PARTY_TYPES)).sum())
funnel = pd.DataFrame([
    {"step": f"registered TPPPs that are TM, party {CUST_PARTY_TYPES}", "TM organisations": None, "TPPPs": tppp_tm_org},
    {"step": f"active in {WINDOW_START} → {WINDOW_END}", "TM organisations": fun["n0"], "TPPPs": fun["t0"]},
    {"step": f"active ≥ {MIN_ACTIVE_MONTHS} months", "TM organisations": fun["n1"], "TPPPs": fun["t1"]},
    {"step": f"≥ {CUST_MIN_AMT:,} received and paid per month", "TM organisations": fun["n2"], "TPPPs": fun["t2"]},
    {"step": f"≥ {CUST_MIN_CPTY} payers or payees per month (screened)", "TM organisations": fun["n3"], "TPPPs": fun["t3"]},
])
funnel["% of TPPPs kept"] = 100 * funnel.TPPPs / max(tppp_tm_org, 1)

C = (cagg_l.where(s3 | F.col("is_tppp") | F.col("tppp_family"))
           .withColumn("screened", s3)
           .join(CDIM.select("mdm_id", "customer_pwr_id", "master_pwr_id", "customer_name", "naics_cd", "naics_desc",
                             "naics_status", name_norm(F.col("customer_name")).alias("name_norm"),
                             name_key(F.col("customer_name")).alias("name_key")), "mdm_id", "left")
           .toPandas())
C["naics6"] = np.where(C.naics_status == "valid", C.naics_cd.astype(str).str[:6], None)
nn = C.name_norm.fillna("")
C["naics_proc"] = C.naics6.isin(PROC_NAICS)
C["name_proc"] = nn.str.contains(PROC_NAME_RE, regex=True)
C["pnc_own"] = nn.str.contains(PNC_NAME_RE, regex=True)
C["bank_like"] = nn.str.contains(BANK_NAME_RE, regex=True) | C.naics6.fillna("").str.startswith(tuple(EXCL_NAICS_PREFIX[:2]))
C["gov"] = C.naics6.fillna("").str.startswith(tuple(p for p in EXCL_NAICS_PREFIX if p.startswith("92")))
C["excluded"] = C.pnc_own | C.bank_like | C.gov
C["naics2v"] = C.naics6.fillna("").str[:2]
_n6 = C.naics6.fillna("")
SECTOR_TPPP = {p: int((C.is_tppp & _n6.str.startswith(p)).sum()) for p in UNLIKELY_NAICS2}
EXCL_SECTORS = sorted(p for p, k in SECTOR_TPPP.items() if k < TPPP_SECTOR_MIN)
C["sector_unlikely"] = _n6.str.startswith(tuple(EXCL_SECTORS)) if EXCL_SECTORS else False
CS = C[C.screened].reset_index(drop=True)          # the scoring population
lost = C[C.is_tppp & ~C.screened]
lost_tbl = lost.assign(**{"$ in /mo": lost.amt_in, "$ out /mo": lost.amt_out})[
    ["customer_name", "naics_cd", "months_active", "$ in /mo", "$ out /mo", "ncpty_in", "ncpty_out"]].head(15)
show(f"§2b Look-alikes population: {len(CS):,} screened TM organisations, {int(CS.is_tppp.sum())} registered TPPPs among them"
     f" ({'built' if rebuilt_c else 'reused'} extract)",
     ("Funnel", funnel),
     note("The screen should keep most TPPPs; if it drops many, loosen CUST_MIN_AMT / CUST_MIN_CPTY."),
     ("Registered TPPPs dropped by the screen (first 15)", lost_tbl) if len(lost) else None)

## §3 What a registered TPPP looks like

Each metric on its own: how well it separates registered TPPPs from the other screened clients (AUC 0.5 = no signal,
1.0 = every TPPP above every other client, 0.0 = every TPPP below).

In [ ]:
# §3  Single-feature separation, registered TPPPs vs screened others
yC = CS.is_tppp.to_numpy(bool)
assert yC.sum() >= 10, f"only {yC.sum()} registered TPPPs in the screened population: loosen the screen"
SEP_C = separation(CS, CUST_FEATS, yC)
tbl = SEP_C.assign(label=SEP_C.feature.map(lambda f: CUST_BY_NAME[f]["label"]),
                   **{"in model": SEP_C.feature.isin(CUST_MODEL_FEATS)})
fig, ax = plt.subplots(figsize=(6.5, 0.22 * len(tbl) + 1))
ax.barh(tbl.label[::-1], tbl.AUC[::-1] - 0.5, left=0.5, color=np.where(tbl.AUC[::-1] > 0.5, "#2a6f97", "#c44e52"))
ax.axvline(0.5, c="grey", lw=0.8)
ax.set_xlim(0, 1)
ax.set_xlabel("AUC: registered TPPPs vs other screened TM organisations")
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/cust_feature_auc.png", dpi=130)
plt.close(fig)
show(f"§3 Registered TPPPs ({int(yC.sum())}) vs other screened TM organisations ({int((~yC).sum()):,})",
     ("Features by separation", tbl[["feature", "label", "AUC", "median examples", "median others",
                                     "% examples with value", "in model"]].rename(
         columns={"median examples": "median TPPP", "median others": "median other"})),
     note(f"Figure: {OUT_DIR}/cust_feature_auc.png"))

## §4 Scoring, in two stages

1. **All screened registered TPPPs as examples.** Each one gets a held-out score (from a model that did not see it).
2. **Reference TPPPs** = those whose held-out score is in the top `REF_MIN_PCT` — the ones that clearly behave like a
   processor in our data. TPPPs that do not (processing through accounts we cannot see) are listed and left out: as
   examples they would pull ordinary businesses into the list. The final models learn from the reference TPPPs only.

Validation is the held-out rank of the reference TPPPs; the non-reference TPPPs are never trained on in stage 2, so
their ranks are honest too. Each client also gets its `N_SIMILAR` most similar reference TPPPs (distance over the
separating metrics) and the metrics on which it is closest to the first of them.

In [ ]:
# §4  Stage 1 (all TPPPs) → reference TPPPs → stage 2 scores, similarity
t0 = time.time()
handle = display(render(note("stage 1…")), display_id=True)
y_all = CS.is_tppp.to_numpy(bool)
fam = CS.tppp_family.to_numpy(bool)
CV_GROUPS = CS.master_pwr_id.fillna(CS.mdm_id).astype(str).to_numpy()     # a corporate family stays in one fold
s1_pct = cv_pct("logit", rank_matrix(CS, CUST_MODEL_FEATS), y_all, ~fam)
REC_S1 = recall_table({"logit, stage 1": s1_pct}, y_all, {"all registered TPPPs": y_all})
CS["stage1_pct"] = s1_pct
ref = y_all & (s1_pct >= REF_MIN_PCT)
if ref.sum() < REF_MIN_N:
    ref = y_all.copy()
CS["is_ref"] = ref
train_ok = ~fam & ~(y_all & ~ref)               # non-reference TPPPs: neither positive nor negative
SEP_R = separation(CS, CUST_FEATS, ref)
handle.update(render(note("stage 2…")))
PCT_C, _, _ = score_population(CS, CUST_MODEL_FEATS, ref, train_ok, SEP_R)
groups = {"reference TPPPs": ref, "other registered TPPPs": y_all & ~ref}
REC_C = recall_table(PCT_C, ref, {k: v for k, v in groups.items() if v.sum()})
FINAL_C = pick_method(REC_C, PCT_C, group="reference TPPPs")
CS["score"] = pd.Series(PCT_C[FINAL_C]).rank(pct=True).to_numpy()
for m_, p_ in PCT_C.items():
    CS[f"pct_{m_}"] = p_
recall_figure(PCT_C, ref, f"{OUT_DIR}/lookalike_recall.png", "Look-alikes: held-out reference TPPPs found vs share reviewed")

# most similar reference TPPPs and shared metrics
pos = np.where(ref)[0]
R = rank_matrix(CS, CUST_MODEL_FEATS)
nidx, ndst = knn_top(R, pos, knn_weights(SEP_R, CUST_MODEL_FEATS), N_SIMILAR)
names = CS.customer_name.fillna("?").to_numpy()
CS["similar_tppps"] = ["; ".join(names[j] for j in row) for row in nidx]
CS["similar_dist"] = ndst[:, 0]
CS["_nn"] = nidx[:, 0]
REF_MED = float(np.median(CS.score[ref]))
CS["screened_rank"] = CS.score.rank(ascending=False, method="min").astype(int)

# archetypes: reference TPPPs grouped by behaviour; each client takes the archetype of its most similar reference TPPP
from sklearn.cluster import KMeans
wk = np.sqrt(knn_weights(SEP_R, CUST_MODEL_FEATS))
k_ = int(min(N_ARCHETYPES, max(1, len(pos) // 4)))
km = KMeans(n_clusters=k_, n_init=20, random_state=SEED).fit(R[pos] * wk)
arch_of_ref = dict(zip(pos, km.labels_ + 1))
arch_rows = []
for a in range(1, k_ + 1):
    mem = [j for j in pos if arch_of_ref[j] == a]
    m_ = CS.iloc[mem]
    top_naics = m_.naics_desc.fillna("unknown").astype(str).str[:44].value_counts()
    lab = f"{a}: {top_naics.index[0]}" if len(top_naics) else str(a)
    cen = pd.Series(km.cluster_centers_[a - 1] / np.where(wk > 0, wk, 1), index=CUST_MODEL_FEATS)
    hi = [CUST_BY_NAME[f]["label"] for f in cen.sort_values(ascending=False).index[:3] if cen[f] >= 0.7]
    arch_rows.append({"archetype": lab, "reference TPPPs": len(mem),
                      "examples": "; ".join(m_.customer_name.fillna("?").head(4)),
                      "typical: high on": ", ".join(hi)})
ARCH = pd.DataFrame(arch_rows)
ARCH_LABEL = dict(zip(range(1, k_ + 1), ARCH.archetype))
# nearest archetype centre (weighted percentile space); a reference TPPP keeps its own cluster
CS["archetype"] = [ARCH_LABEL[a + 1] for a in km.predict(R * wk)]
for j in pos:
    CS.iat[j, CS.columns.get_loc("archetype")] = ARCH_LABEL[arch_of_ref[j]]
CS["as_tppp_like"] = CS.score >= REF_MED        # at least as TPPP-like as the typical reference TPPP

reft = CS[y_all].sort_values("stage1_pct", ascending=False)
ref_tbl = pd.DataFrame({"registered TPPP": reft.customer_name, "NAICS": reft.naics_cd, "held-out pct (stage 1)": 100 * reft.stage1_pct,
                        "reference": reft.is_ref, "payers /mo": reft.ncpty_in.round(), "payees /mo": reft.ncpty_out.round()})
handle.update(render(
    f"§4 Look-alike scores ({(time.time() - t0) / 60:.1f} min): {int(ref.sum())} reference TPPPs of {int(y_all.sum())} screened",
    ("Honest recall: stage 1 over all registered TPPPs (none was selected for scoring high)", REC_S1),
    ("Stage 2 (reference TPPPs were chosen for scoring high in stage 1, so this is flattered)", REC_C),
    ("Archetypes of reference TPPPs (behaviour groups)", ARCH),
    note(f"List ranks by '{FINAL_C}'. The typical reference TPPP scores {100 * REF_MED:.1f}; "
         f"{int((CS.as_tppp_like & ~CS.is_tppp).sum()):,} unregistered clients score at least that high."),
    ("Registered TPPPs: stage-1 held-out percentile (reference = top " + f"{100 * (1 - REF_MIN_PCT):g}%)", ref_tbl),
    note(f"Figure: {OUT_DIR}/lookalike_recall.png")))

## §5 Look-alike list

Unregistered TM organisations ranked by score. Left off: registered TPPPs, their corporate family, PNC's own entities.
Industry, a *declared* flag (NAICS or name says processor) and a *bank / government* flag are columns for the
reviewer, not filters.

In [ ]:
# §5  Look-alike list
_n = CS.naics_cd.fillna("").astype(str)
_ref_n = _n[CS.is_ref]
EXCL_USED = [p_ for p_ in LIST_EXCL_NAICS if not _ref_n.str.startswith(p_).any()]
EXCL_KEPT = [p_ for p_ in LIST_EXCL_NAICS if p_ not in EXCL_USED]
CS["sector_excluded"] = _n.str.startswith(tuple(EXCL_USED)) if EXCL_USED else False
cand = CS[~CS.is_tppp & ~CS.tppp_family & ~CS.pnc_own & ~CS.sector_excluded].sort_values("score", ascending=False)
if ARCHETYPE_SHOW:
    keep_ = {ARCH_LABEL[a] for a in ARCHETYPE_SHOW}
    cand = cand[cand.archetype.isin(keep_)]
LIST_C = cand.head(TOP_N_CUST).copy()
LIST_C.insert(0, "rank", np.arange(1, len(LIST_C) + 1))
pos_in = {i: k for k, i in enumerate(CS.index)}
_prep = shared_prep(CS, CUST_MODEL_FEATS, SEP_R)
LIST_C["shared_metrics"] = [shared_metrics(_prep, CUST_BY_NAME, pos_in[i], int(CS.at[i, "_nn"])) for i in LIST_C.index]
LIST_C["declared"] = LIST_C.naics_proc | LIST_C.name_proc
LIST_C["flag"] = np.where(LIST_C.bank_like, "bank-like", np.where(LIST_C.gov, "government", ""))

def look_view(d):
    return pd.DataFrame({
        "rank": d["rank"], "client": d.customer_name, "power id": d.customer_pwr_id,
        "naics": d.naics_cd.fillna("").astype(str) + " " + d.naics_desc.fillna("").astype(str).str[:30],
        "archetype": d.archetype, "declared": d.declared, "flag": d.flag, "screened rank": d.screened_rank,
        "payers /mo": d.ncpty_in.round(), "payees /mo": d.ncpty_out.round(),
        "$K in /mo": d.amt_in_st / 1e3, "$K out /mo": d.amt_out_st / 1e3,
        "most similar registered TPPPs": d.similar_tppps, "closest on": d.shared_metrics})

n_like = int((cand.score >= REF_MED).sum())
show(f"§5 Look-alikes: {n_like:,} unregistered clients are at least as TPPP-like as the typical reference TPPP",
     ("Archetypes among the top " + str(len(LIST_C)), LIST_C.archetype.value_counts().rename_axis("archetype").reset_index(name="look-alikes")),
     (f"Top {SHOW_N}", look_view(LIST_C.head(SHOW_N))),
     *[(f"Top 10 resembling archetype {a}", look_view(LIST_C[LIST_C.archetype == a].head(10))) for a in ARCH.archetype],
     ("Declared processors among the top 300", pd.DataFrame({"declared (NAICS or name)": [int(LIST_C.declared.sum())],
                                                            "not declared": [int((~LIST_C.declared).sum())]})),
     note("'closest on' = metrics where the client and its most similar registered TPPP are both in the TPPP-like "
          "quarter and closest to each other (client value vs the TPPP's)."),
     note(f"Left off the list by sector ({', '.join(EXCL_USED) or 'none'}): "
          f"{int((CS.sector_excluded & ~CS.is_tppp).sum()):,} clients, kept in the scores table."
          + (f" Kept because a reference TPPP sits there: {', '.join(EXCL_KEPT)}." if EXCL_KEPT else "")))

# Task 2 — Prospects: counterparties behaving like TPPPs

## §6 Counterparties, examples and screen

Counterparty metrics (`pkg_cpty_metrics_monthly`), monthly averages over the window. Direction is the counterparty's
view: `_in` = it receives from PNC clients.

**Examples**

| Set | Rule |
|---|---|
| `tppp_outside` | registered TPPPs' accounts at other banks: same cleaned name as a registered TPPP (`exact`), or the TPPP's distinctive first name word (`brand`, e.g. PAYCHEX EIB, ADP 401K; the word is shared by ≤ `BRAND_MAX_FAM` PNC families) |
| `known` | well-known processors by name (`KNOWN_PROC_RE`), not already in `tppp_outside` |

**Relationship to PNC** (prospects must not already bank with us): `existing PNC client` = same cleaned name as a PNC
organisation; `possible PNC client` = shares a distinctive first word with ≤ `LOOSE_MAX_FREQ` PNC families (flagged,
kept); otherwise `new to bank`.

In [ ]:
# §6a  Counterparty feature spec → table columns
CPTY_SPEC = [
    *both("amt", "amt", "$ from PNC clients /mo", "$ to PNC clients /mo", "size", "usd"),
    *both("ncust", "n_cust", "PNC clients paying it", "PNC clients it pays", fmt="int"),
    *both("ncust_tm", "n_cust_tm", "TM clients paying it", "TM clients it pays", fmt="int"),
    *both("org", "org_share", "$ in from organisations", "$ out to organisations", fmt="pct"),
    *both("tm", "tm_share", "$ in from TM clients", "$ out to TM clients", fmt="pct"),
    *both("top1", "top1_share", "largest payer share", "largest payee share", fmt="pct"),
    *both("naics_ent", "naics2_entropy", "industry spread of payers", "industry spread of payees"),
    *both("dep", "dep_wavg", "share of payers' outside flow", "share of payees' outside flow", fmt="pct"),
    feat("two_way", ["two_way_ratio"], "two-way balance", fmt="ratio"),
    *both("days", "active_days", "days with money in /mo", "days with money out /mo"),
    *both("ticket", "avg_ticket", "average payment in", "average payment out", fmt="usd"),
    *both("retention", "retention", "payer retention", "payee retention", fmt="pct"),
    *both("rec", "rec_standing", "recurring payers", "recurring payees", fmt="int"),
    *both("new_amt", "new_amt_share", "$ in from new payers", "$ out to new payees", fmt="pct"),
    *both("ach", "share_ach", "ACH share in", "ACH share out", fmt="pct"),
    *both("wire", "share_wire", "wire share in", "wire share out", fmt="pct"),
    *both("rails", "n_rails", "rails in", "rails out"),
    feat("nfi", ["n_fi"], "banks holding its accounts", fmt="int"),
    feat("topfi", ["top_fi_share"], "share at its main bank", fmt="pct"),
    feat("nids", ["n_ids_active"], "account ids active", fmt="int"),
    feat("ncust_all", ["n_cust_all"], "PNC clients either way", fmt="int", model=False),
]
CPTY_DERIVED = [feat("ach_orig", [], "months as ACH originator", fmt="pct"),
                feat("role_flip", [], "org share in − out", fmt="ratio")]
CPTY_BY_NAME = {f["name"]: f for f in CPTY_SPEC + CPTY_DERIVED}
CPTY_DICT = read_dict(T_CPTY_DICT)
CPTY_COL, pres = resolve(CPTY_SPEC, spark.table(T_CPTY_MET).columns, CPTY_DICT)
miss_core = [f for f in ["amt_in", "amt_out", "ncust_in", "ncust_out", "ncust_all"] if f not in CPTY_COL]
assert not miss_core, f"core counterparty features missing: {miss_core}"
need = {"ent_id", "ent_name", "name_class", "cpty_type", "fi_dom", "eligible", "amt_all", "month"}
assert need <= set(spark.table(T_CPTY_MET).columns), f"{T_CPTY_MET} lacks {need - set(spark.table(T_CPTY_MET).columns)}"
miss_m = [m for m in WINDOW if m not in partitions(T_CPTY_MET)]
assert not miss_m, f"{T_CPTY_MET}: window months missing {miss_m}"
CPTY_FEATS = ([f["name"] for f in CPTY_SPEC if f["model"] and f["name"] in CPTY_COL]
              + (["ach_orig"]) + (["role_flip"] if {"org_in", "org_out"} <= set(CPTY_COL) else []))
CPTY_MODEL_FEATS = [f for f in CPTY_FEATS if SCORE_SET == "full" or CPTY_BY_NAME[f]["kind"] != "size"]
show(f"§6a Counterparty features: {len(CPTY_COL)} of {len(CPTY_SPEC)} resolved, {len(CPTY_MODEL_FEATS)} used by the models",
     ("Resolution", pres),
     note("Derived: ach_orig = share of active months with cpty_type NON_PNC_ACH_ORIGINATOR; "
          "role_flip = org share in − org share out (payroll: in from employers, out to people)."))

In [ ]:
# §6b  Window averages per counterparty (written once), examples, relationship to PNC, screen
cols_p = sorted(set(CPTY_COL.values()))
SIG_P = hashlib.md5(json.dumps({"v": VERSION, "w": WINDOW, "s": STABLE_END, "c": cols_p, "k": CPTY_CLASSES}).encode()).hexdigest()[:10]

def build_cpty_agg():
    m = (spark.table(T_CPTY_MET).where(F.col("month").isin(WINDOW))
              .where(F.col("eligible") & F.col("name_class").isin(CPTY_CLASSES) & (F.col("amt_all") > 0))
              .select("ent_id", "month", "ent_name", "name_class", "cpty_type", "fi_dom",
                      F.col("amt_all").cast("double").alias("_amt"),
                      *[F.col(c).cast("double").alias(c) for c in cols_p]))
    aggs = [F.count("*").alias("months_active"),
            F.max(F.struct("_amt", "ent_name")).getField("ent_name").alias("ent_name"),
            F.max(F.struct("month", "name_class")).getField("name_class").alias("name_class"),
            F.max(F.struct("month", "fi_dom")).getField("fi_dom").alias("fi_dom"),
            F.avg((F.col("cpty_type") == "NON_PNC_ACH_ORIGINATOR").cast("double")).alias("ach_orig"),
            F.avg(F.when(F.col("month").isin(STABLE), F.col("_amt"))).alias("amt_st")]
    aggs += [F.avg(F.col(CPTY_COL[f])).alias(f) for f in CPTY_COL]
    return (m.groupBy("ent_id").agg(*aggs)
             .withColumn("name_norm", name_norm(F.col("ent_name")))
             .withColumn("name_key", name_key(F.col("ent_name")))
             .withColumn("fw", first_word("name_key")))

pagg, rebuilt_p = extract(f"{P_TMP}/cpty_agg_v6", SIG_P, build_cpty_agg)

# ---- PNC organisations: names, first words, families ----
orgs = (CDIM.where(F.col("party_type") == "O")
            .select("mdm_id", "customer_name", "is_tm", F.coalesce("master_pwr_id", "mdm_id").alias("fam"),
                    name_key(F.col("customer_name")).alias("name_key"))
            .where(F.col("name_key").isNotNull()).withColumn("fw", first_word("name_key")))
fw_stat = orgs.groupBy("fw").agg(F.countDistinct("fam").alias("fam_n")).cache()
tp = orgs.where(F.col("mdm_id").isin(list(TPPP_IDS)))
exact_t = tp.groupBy("name_key").agg(F.first("customer_name").alias("tppp_exact"))
tp_b = tp.where(~name_norm(F.col("customer_name")).rlike(BRAND_BANK_RE) & ~F.col("fw").isin(BRAND_EXCLUDE))
brand_t = (tp_b.select("fw", "customer_name").join(fw_stat.where(F.col("fam_n") <= BRAND_MAX_FAM), "fw")
               .groupBy("fw").agg(F.first("customer_name").alias("tppp_brand")))
brand_any = tp_b.groupBy("fw").agg(F.first("customer_name").alias("tppp_brand_any"))   # for known processors
ex_org = orgs.groupBy("name_key").agg(F.count("*").alias("n_exact"), F.expr("max_by(customer_name, cast(is_tm as int))").alias("exact_name"))
# possible PNC client: same rare first word AND one name's distinctive words (one of ≥ 4 letters) all in the other's
toks = lambda c: F.expr(f"array_distinct(filter(split({c}, ' '), t -> length(t) >= 3 AND NOT array_contains(array({stop_sql}), t)))")
cand_pairs = (pagg.select("ent_id", "fw", toks("name_key").alias("ta"))
                  .join(orgs.join(fw_stat.where(F.col("fam_n") <= LOOSE_MAX_FREQ), "fw")
                            .select("fw", "customer_name", toks("name_key").alias("tb")), "fw")
                  .withColumn("common", F.array_intersect("ta", "tb"))
                  .where(((F.size(F.array_except("ta", "tb")) == 0) | (F.size(F.array_except("tb", "ta")) == 0))
                         & F.expr("exists(common, t -> length(t) >= 4)")))
lo_ent = cand_pairs.groupBy("ent_id").agg(
    F.concat_ws("; ", F.slice(F.sort_array(F.collect_set("customer_name")), 1, 3)).alias("loose_names"))
pl = (pagg.join(F.broadcast(exact_t), "name_key", "left").join(F.broadcast(brand_t), "fw", "left")
          .join(F.broadcast(brand_any), "fw", "left")
          .join(ex_org, "name_key", "left").join(lo_ent, "ent_id", "left")
          .withColumn("known_proc", F.col("name_norm").rlike(KNOWN_PROC_RE))
          .withColumn("tppp_kbrand", F.when(F.col("known_proc") & F.col("tppp_exact").isNull()
                                            & F.col("tppp_brand").isNull(), F.col("tppp_brand_any")))
          .withColumn("tppp_outside", F.col("tppp_exact").isNotNull() | F.col("tppp_brand").isNotNull()
                      | F.col("tppp_kbrand").isNotNull())
          .withColumn("tppp_match", F.when(F.col("tppp_exact").isNotNull(), F.lit("exact"))
                                     .when(F.col("tppp_brand").isNotNull(), F.lit("brand"))
                                     .when(F.col("tppp_kbrand").isNotNull(), F.lit("known brand")))
          .withColumn("tppp_name", F.coalesce("tppp_exact", "tppp_brand", "tppp_kbrand"))
          .withColumn("cpty_excl", F.col("name_norm").rlike(CPTY_EXCL_RE)))
q1 = F.col("months_active") >= MIN_ACTIVE_MONTHS
q2a = q1 & (F.col("ncust_all") >= CPTY_MIN_CUST)
org_max = F.greatest(F.coalesce(F.col("org_in"), F.lit(0.0)), F.coalesce(F.col("org_out"), F.lit(0.0)))
q2 = q2a & (org_max >= CPTY_MIN_ORG_SHARE) & ~F.col("cpty_excl")
steps = [("counterparties with full metrics, active in the window", F.lit(True)),
         (f"active ≥ {MIN_ACTIVE_MONTHS} months", q1), (f"≥ {CPTY_MIN_CUST} PNC clients per month", q2a),
         (f"≥ {100 * CPTY_MIN_ORG_SHARE:g}% of $ with organisations on one side, not excluded (screened)", q2)]
is_ex = F.col("tppp_match") == "exact"
is_br = F.col("tppp_match").isin("brand", "known brand")
is_kn = F.col("known_proc") & ~F.col("tppp_outside")
fr = pl.agg(*[F.sum(c.cast("int")).alias(f"n{i}") for i, (_, c) in enumerate(steps)],
            *[F.sum((c & is_ex).cast("int")).alias(f"e{i}") for i, (_, c) in enumerate(steps)],
            *[F.sum((c & is_br).cast("int")).alias(f"b{i}") for i, (_, c) in enumerate(steps)],
            *[F.sum((c & is_kn).cast("int")).alias(f"k{i}") for i, (_, c) in enumerate(steps)]).first()
pfun = pd.DataFrame([{"step": s, "counterparties": fr[f"n{i}"], "TPPP outside: exact": fr[f"e{i}"],
                      "TPPP outside: brand": fr[f"b{i}"], "known processors": fr[f"k{i}"]} for i, (s, _) in enumerate(steps)])
P = pl.where(q2).toPandas()
if {"org_in", "org_out"} <= set(P.columns):
    P["role_flip"] = P.org_in - P.org_out
P["ex_tppp"] = P.tppp_outside & ~P.cpty_excl
P["ex_known"] = P.known_proc & ~P.tppp_outside & ~P.cpty_excl
P["example"] = P.ex_tppp | P.ex_known
P["relationship"] = np.select([P.n_exact.fillna(0) > 0, P.loose_names.notna()],
                              ["existing PNC client", "possible PNC client"], default="new to bank")
P["pnc_match_names"] = P.exact_name.fillna(P.loose_names)
_tok = P.name_norm.fillna("").map(lambda n: [t for t in n.split() if len(t) > 2 and not re.fullmatch(r"[A-Z]*\d+[A-Z\d]*", t)])
P["generic_name"] = _tok.map(lambda t: len([x for x in t if x not in GENERIC_TOKENS]) == 0)
outside = P[P.ex_tppp].sort_values("ncust_all", ascending=False)
show(f"§6b Prospects population: {len(P):,} screened counterparties; examples: {int(P.ex_tppp.sum())} TPPP outside "
     f"accounts, {int(P.ex_known.sum())} known processors ({'built' if rebuilt_p else 'reused'} extract)",
     ("Funnel", pfun),
     ("Registered TPPPs' outside accounts that passed the screen", pd.DataFrame({
         "outside account": outside.ent_name, "registered TPPP": outside.tppp_name, "match": outside.tppp_match,
         "bank today": outside.fi_dom, "PNC clients paying": outside.ncust_in.round(),
         "PNC clients paid": outside.ncust_out.round(), "$K /mo with PNC clients": outside.amt_st / 1e3}).head(40)),
     note("Brand matches can catch an unrelated company sharing the first word; check the list above once."))

## §7 What an outside TPPP looks like

Each counterparty metric on its own, for the two example sets against the other screened counterparties. If the
TPPP outside accounts and the known processors separate on the same metrics, the two sets describe the same kind of
company and can be combined; if not, the outside accounts are the closer picture of a prospect.

In [ ]:
# §7  Profiles of the two example sets
others = ~P.example.to_numpy(bool)
y_t, y_k = P.ex_tppp.to_numpy(bool), P.ex_known.to_numpy(bool)
def sep_vs_others(y):
    m = y | others
    return separation(P[m].reset_index(drop=True), CPTY_FEATS, y[m]).set_index("feature")
ST, SK = sep_vs_others(y_t), sep_vs_others(y_k)
prof = pd.DataFrame({
    "label": [CPTY_BY_NAME[f]["label"] for f in CPTY_FEATS],
    "AUC TPPP outside": ST.reindex(CPTY_FEATS).AUC.values, "AUC known": SK.reindex(CPTY_FEATS).AUC.values,
    "median TPPP outside": ST.reindex(CPTY_FEATS)["median examples"].values,
    "median known": SK.reindex(CPTY_FEATS)["median examples"].values,
    "median others": ST.reindex(CPTY_FEATS)["median others"].values}, index=CPTY_FEATS)
prof["max separation"] = np.maximum((prof["AUC TPPP outside"] - 0.5).abs(), (prof["AUC known"] - 0.5).abs())
prof = prof.sort_values("max separation", ascending=False).reset_index().rename(columns={"index": "feature"})
ok_ = prof[["AUC TPPP outside", "AUC known"]].dropna()
agree = float(ok_.corr(method="spearman").iloc[0, 1]) if len(ok_) > 3 else np.nan
same_dir = float(((ok_["AUC TPPP outside"] - 0.5) * (ok_["AUC known"] - 0.5) > 0).mean()) if len(ok_) else np.nan
show(f"§7 Profiles: {int(y_t.sum())} TPPP outside accounts, {int(y_k.sum())} known processors, {int(others.sum()):,} others",
     ("Metrics by separation (either set)", prof.drop(columns=["max separation"]).head(25)),
     note(f"Agreement between the two sets: rank correlation of their AUCs {agree:.2f}; the same direction on "
          f"{100 * same_dir:.0f}% of metrics. High agreement → combining them is safe."))

## §8 Scoring and validation

Two example choices × three methods, all out of fold:

| Variant | Examples (positives) | Known processors in training |
|---|---|---|
| `tppp_outside` | TPPP outside accounts | left out (neither positive nor negative) |
| `combined` | TPPP outside accounts + known processors | positives |

Each variant is tested on how high hidden TPPP outside accounts and hidden known processors come back. The list uses
the variant and method with the best median rank on TPPP outside accounts (when at least `MIN_EX_TPPP` are screened;
otherwise on all examples).

In [ ]:
# §8  Out-of-fold scores for both variants; choose; similarity and reasons
t0 = time.time()
handle = display(render(note("scoring…")), display_id=True)
VARIANTS = {"tppp_outside": (y_t, ~y_k), "combined": (y_t | y_k, np.ones(len(P), bool))}
if y_t.sum() < 5:
    VARIANTS.pop("tppp_outside")
_grp = np.where(P.ex_tppp, "T:" + P.tppp_name.fillna(P.ent_name).astype(str),
                np.where(P.ex_known, "K:" + P.fw.fillna(P.ent_name).astype(str), "E:" + P.ent_id.astype(str)))
CV_GROUPS = _grp                                  # all accounts of one company stay in one fold
RES, recs = {}, []
for vname, (yv, tok) in VARIANTS.items():
    handle.update(render(note(f"variant {vname}…")))
    sep_v = separation(P, CPTY_FEATS, yv)
    pct_v, near_v, dist_v = score_population(P, CPTY_MODEL_FEATS, yv, tok, sep_v)
    RES[vname] = (pct_v, near_v, sep_v)
    g = {"TPPP outside accounts": y_t, "known processors": y_k}
    r = recall_table(pct_v, yv, {k: v for k, v in g.items() if v.sum()})
    recs.append(r.assign(variant=vname))
REC_P = pd.concat(recs)[["variant"] + [c for c in recs[0].columns if c != "variant"]]
grp = "TPPP outside accounts" if y_t.sum() >= MIN_EX_TPPP else None
cand_rows = REC_P[REC_P.method.isin(METHODS + ["blend"])]
if grp:
    cand_rows = cand_rows[cand_rows.examples == grp]
else:
    cand_rows = cand_rows.groupby(["variant", "method"], as_index=False)["median % ranked above"].mean()
if FINAL_METHOD != "auto":
    cand_rows = cand_rows[cand_rows.method == FINAL_METHOD]
best_ = cand_rows.sort_values("median % ranked above").iloc[0]
FINAL_V, FINAL_P = best_.variant, best_.method
pct_f, near_f, SEP_F = RES[FINAL_V]
P["score"] = pd.Series(pct_f[FINAL_P]).rank(pct=True).to_numpy()
P["screened_rank"] = P.score.rank(ascending=False, method="min").astype(int)
for m_, p_ in pct_f.items():
    P[f"pct_{m_}"] = p_
yf = VARIANTS[FINAL_V][0]
pos_p = np.where(yf)[0]
nidx_p, _ = knn_top(rank_matrix(P, CPTY_MODEL_FEATS), pos_p, knn_weights(SEP_F, CPTY_MODEL_FEATS), 1)
lab_ = np.where(P.ex_tppp, P.tppp_name.fillna(P.ent_name) + " (TPPP outside)", P.ent_name + " (known)")
P["most_similar_example"] = lab_[nidx_p[:, 0]]
P["reasons"] = reasons(P, CPTY_MODEL_FEATS, SEP_F, CPTY_BY_NAME)
recall_figure({k: v for k, v in pct_f.items()}, yf, f"{OUT_DIR}/prospect_recall.png",
              f"Prospects ({FINAL_V}): held-out examples found vs share reviewed")
EX_MED = float(np.median(P.score[P.ex_tppp])) if P.ex_tppp.any() else float(np.median(P.score[yf]))
handle.update(render(
    f"§8 Prospect scores ({(time.time() - t0) / 60:.1f} min): list uses variant '{FINAL_V}', method '{FINAL_P}'",
    ("Held-out recall by variant and kind of example", REC_P),
    note("Read across a row: how high hidden examples of each kind come back. If 'tppp_outside' finds the known "
         "processors well, the outside accounts are a good picture of processors in general."),
    note(f"The typical TPPP outside account scores {100 * EX_MED:.1f}. Figure: {OUT_DIR}/prospect_recall.png")))

## §9 Prospect list

Counterparties ranked by score. Left off: registered TPPPs' outside accounts (they are our clients' accounts),
`existing PNC client`, excluded names (networks, government, issuers, insurers) and generic descriptors (MERCHANT
SERVICE, MERCH BNKCD … — no company name to research). Known processors stay, flagged. Size columns (money with PNC
clients up to `STABLE_END`, PNC and TM clients) and the three TM clients moving the most money with each prospect
(warm introductions) are for the reviewer; they do not change the order.

In [ ]:
# §9  Prospect list with warm introductions
pool = P[~P.ex_tppp & ~P.cpty_excl & ~P.generic_name & (P.relationship != "existing PNC client")]
LIST_P = pool.sort_values("score", ascending=False).head(TOP_N_CPTY).copy()
LIST_P.insert(0, "rank", np.arange(1, len(LIST_P) + 1))
pc = {c.lower(): c for c in spark.table(T_PAIR).columns}
P_CUST, P_DIR, P_SCOPE, P_OTHER, P_AMT = (pc.get("cust", pc.get("mdm_id")), pc["dir"], pc["scope"],
                                          pc.get("other", pc.get("other_id")), pc.get("amount", pc.get("amt")))
pr = (spark.table(T_PAIR).where(F.col("month").isin(WINDOW) & (F.col(P_SCOPE) == "ext"))
           .join(F.broadcast(spark.createDataFrame(LIST_P[["ent_id"]])), F.col(P_OTHER) == F.col("ent_id"))
           .groupBy("ent_id", F.col(P_CUST).alias("mdm_id"))
           .agg(F.sum(F.when(F.col(P_DIR) == "out", F.col(P_AMT))).cast("double").alias("paid"),
                F.sum(F.when(F.col(P_DIR) == "in", F.col(P_AMT))).cast("double").alias("received"))
           .join(CDIM.where("is_tm").select("mdm_id", "customer_name", "customer_pwr_id"), "mdm_id"))
INTRO = pr.toPandas().fillna({"paid": 0.0, "received": 0.0})
INTRO["total"] = INTRO.paid + INTRO.received
INTRO = INTRO.sort_values(["ent_id", "total"], ascending=[True, False])
INTRO["intro_rank"] = INTRO.groupby("ent_id").cumcount() + 1
n_tm = INTRO.groupby("ent_id").mdm_id.nunique()
top3 = {e: "; ".join(f"{n} ({t / 1e3:,.0f}K)" for n, t in zip(d.customer_name, d.total))
        for e, d in INTRO[INTRO.intro_rank <= INTRO_TOP].groupby("ent_id")}
LIST_P["tm_clients"] = LIST_P.ent_id.map(n_tm).fillna(0).astype(int)
LIST_P["warm_intros"] = LIST_P.ent_id.map(top3).fillna("")

def pros_view(d):
    return pd.DataFrame({
        "rank": d["rank"], "counterparty": d.ent_name, "relationship": d.relationship,
        "PNC name match": d.pnc_match_names, "known processor": d.known_proc, "bank today": d.fi_dom,
        "banks": d.nfi.round(1) if "nfi" in d else np.nan, "screened rank": d.screened_rank,
        "$K /mo with PNC clients": d.amt_st / 1e3, "PNC clients paying": d.ncust_in.round(),
        "PNC clients paid": d.ncust_out.round(), "TM clients (window)": d.tm_clients,
        "most similar example": d.most_similar_example, "why": d.reasons, "warm introductions": d.warm_intros})

n_like_p = int((pool.score >= EX_MED).sum())
show(f"§9 Prospects: {len(pool):,} candidates; {n_like_p:,} at least as TPPP-like as the typical example",
     ("Relationship to PNC, top " + str(len(LIST_P)), LIST_P.relationship.value_counts().rename_axis("relationship").reset_index(name="prospects")),
     (f"Top {SHOW_N}", pros_view(LIST_P.head(SHOW_N))),
     (f"Top {SHOW_N} not on the known-processor list", pros_view(LIST_P[~LIST_P.known_proc].head(SHOW_N))))

## §10 Outputs

Tables and review CSVs (with empty `verdict`, `note`, `reviewer`, `reviewed_at`: reviewed rows become examples for
the next version), and the **de-identified enrichment file** for prospects: a random-looking row id and the name only.
The key that joins row ids back stays inside PNC. Get sign-off before any file leaves the bank.

In [ ]:
# §10  Write tables, CSVs, enrichment export; summary
def tidy(d, keep):
    return d[[c for c in keep if c in d.columns]].copy()

OUT_C = tidy(CS, ["mdm_id", "customer_pwr_id", "master_pwr_id", "customer_name", "naics_cd", "naics_desc", "months_active",
                  "is_tppp", "is_ref", "tppp_family", "pnc_own", "bank_like", "gov", "naics_proc", "name_proc",
                  "sector_excluded", "archetype", "screened_rank",
                  "stage1_pct", "score", "as_tppp_like", "similar_tppps", "similar_dist", "amt_in_st", "amt_out_st"]
              + [c for c in CS.columns if c.startswith("pct_")] + list(CUST_COL)).assign(
          window=f"{WINDOW_START}..{WINDOW_END}", version=VERSION)
OUT_P = tidy(P, ["ent_id", "ent_name", "name_class", "fi_dom", "months_active", "tppp_outside", "tppp_match", "tppp_name",
                 "known_proc", "cpty_excl", "ex_tppp", "ex_known", "relationship", "pnc_match_names", "generic_name",
                 "score", "most_similar_example", "reasons", "amt_st"]
              + [c for c in P.columns if c.startswith("pct_")] + CPTY_FEATS + ["ncust_all"]).assign(
          window=f"{WINDOW_START}..{WINDOW_END}", version=VERSION, variant=FINAL_V, method=FINAL_P)
written = []
if WRITE_TABLES:
    for pdf, name in [(OUT_C, "pkg_tppp_lookalikes_v8"), (OUT_P, "pkg_tppp_prospects_v8")]:
        write_table(pdf, name)
        written.append({"table": f"{DB}.{name}", "rows": len(pdf)})
review = {"verdict": "", "note": "", "reviewer": "", "reviewed_at": ""}
f_c, f_p = f"{OUT_DIR}/tppp_lookalikes_v8.csv", f"{OUT_DIR}/tppp_prospects_v8.csv"
look_view(LIST_C).assign(mdm_id=LIST_C.mdm_id.values, **review).to_csv(f_c, index=False)
pros_view(LIST_P).assign(ent_id=LIST_P.ent_id.values, **review).to_csv(f_p, index=False)
REC_C.to_csv(f"{OUT_DIR}/lookalike_recall.csv", index=False)
REC_P.to_csv(f"{OUT_DIR}/prospect_recall.csv", index=False)
prof.to_csv(f"{OUT_DIR}/outside_tppp_profile.csv", index=False)

def rid(x):
    return hashlib.sha256(f"{VERSION}|{x}".encode()).hexdigest()[:12]
xb = pd.DataFrame({"row_id": LIST_P.ent_id.map(rid), "name": LIST_P.ent_name})
f_x, f_k = f"{OUT_DIR}/enrich_prospects_names_v8.csv", f"{OUT_DIR}/enrich_key_INTERNAL_v8.csv"
xb.to_csv(f_x, index=False)
pd.DataFrame({"row_id": xb.row_id, "ent_id": LIST_P.ent_id.values}).to_csv(f_k, index=False)

def found(rec, group, k=0.01, extra=None):
    r = rec[(rec.examples == group)]
    if extra is not None:
        r = r[extra(r)]
    col = next(c for c in rec.columns if c.startswith(f"found in top {100 * k:g}%"))
    return f"{int(r[col].iloc[0])} of {int(r.n.iloc[0])}" if len(r) else "–"
head = pd.DataFrame([
    {"item": "Task 1 — registered TPPPs: screened / reference", "value": f"{int(CS.is_tppp.sum())} / {int(CS.is_ref.sum())}"},
    {"item": f"Task 1 — hidden reference TPPPs found in the top 1% ({FINAL_C})",
     "value": found(REC_C, "reference TPPPs", extra=lambda r: r.method == FINAL_C)},
    {"item": "Task 1 — unregistered clients at least as TPPP-like as the typical reference TPPP", "value": f"{n_like:,}"},
    {"item": "Task 2 — screened counterparties / TPPP outside accounts / known processors",
     "value": f"{len(P):,} / {int(P.ex_tppp.sum())} / {int(P.ex_known.sum())}"},
    {"item": f"Task 2 — hidden TPPP outside accounts found in the top 5% ({FINAL_V}, {FINAL_P})",
     "value": found(REC_P, "TPPP outside accounts", 0.05, lambda r: (r.method == FINAL_P) & (r.variant == FINAL_V))},
    {"item": "Task 2 — candidates at least as TPPP-like as the typical example", "value": f"{n_like_p:,}"},
    {"item": f"Task 2 — top {len(LIST_P)}: new to bank / possible PNC client",
     "value": f"{int((LIST_P.relationship == 'new to bank').sum())} / {int((LIST_P.relationship == 'possible PNC client').sum())}"},
])
show("§10 Summary",
     ("Headline", head),
     ("Tables written", pd.DataFrame(written)) if written else note("WRITE_TABLES = False"),
     ("Files", pd.DataFrame({"file": [f_c, f_p, f_x, f_k + "  (never share)", f"{OUT_DIR}/lookalike_recall.png",
                                      f"{OUT_DIR}/prospect_recall.png", f"{OUT_DIR}/outside_tppp_profile.csv"]})),
     note("Impala: " + " ".join(f"INVALIDATE METADATA {w['table']};" for w in written)) if written else None)

## §11 Review workbook

One Excel file for the team review: **README** (what the lists are, how to fill the verdict), **Look-alikes**,
**Prospects**, **KYC check** (clients whose industry code or name says processor, not registered, not on the
look-alike list), **Archetypes**. Technical columns are left out; `verdict`, `note`, `reviewer`, `reviewed_at` are
empty for the reviewers. It names clients: share it only in the working group's access-controlled folder.

In [ ]:
# §11  Review workbook (Excel)
f_xl = f"{OUT_DIR}/tppp_review_v8.xlsx"
VERDICTS = "processor / not a processor / unsure"
lk = LIST_C.head(REVIEW_N)
wb_look = pd.DataFrame({
    "rank": lk["rank"].values, "client": lk.customer_name.values, "power id": lk.customer_pwr_id.values,
    "industry": (lk.naics_cd.fillna("").astype(str) + " " + lk.naics_desc.fillna("").astype(str)).values,
    "looks like (archetype)": lk.archetype.values, "industry says processor": lk.declared.map({True: "yes", False: ""}).values,
    "payers per month": lk.ncpty_in.round().values, "payees per month": lk.ncpty_out.round().values,
    "received per month ($K)": (lk.amt_in_st / 1e3).round().values, "paid per month ($K)": (lk.amt_out_st / 1e3).round().values,
    "most similar registered TPPPs": lk.similar_tppps.values, "closest on": lk.shared_metrics.values,
    "verdict": "", "note": "", "reviewer": "", "reviewed_at": ""})
pp = LIST_P.head(REVIEW_N)
wb_pros = pd.DataFrame({
    "rank": pp["rank"].values, "company": pp.ent_name.values, "relationship to PNC": pp.relationship.values,
    "PNC name match (check)": pp.pnc_match_names.fillna("").values,
    "known processor": pp.known_proc.map({True: "yes", False: ""}).values, "bank today": pp.fi_dom.values,
    "$K per month with PNC clients": (pp.amt_st / 1e3).round().values,
    "PNC clients paying it": pp.ncust_in.round().values, "PNC clients it pays": pp.ncust_out.round().values,
    "TM clients": pp.tm_clients.values, "most similar example": pp.most_similar_example.values,
    "why": pp.reasons.values, "warm introductions": pp.warm_intros.values,
    "verdict": "", "note": "", "reviewer": "", "reviewed_at": ""})
kyc = CS[(CS.naics_proc | CS.name_proc) & ~CS.is_tppp & ~CS.tppp_family & ~CS.mdm_id.isin(LIST_C.mdm_id)] \
        .sort_values("amt_in_st", ascending=False).head(REVIEW_N)
wb_kyc = pd.DataFrame({
    "client": kyc.customer_name.values, "power id": kyc.customer_pwr_id.values,
    "industry": (kyc.naics_cd.fillna("").astype(str) + " " + kyc.naics_desc.fillna("").astype(str)).values,
    "received per month ($K)": (kyc.amt_in_st / 1e3).round().values,
    "payers per month": kyc.ncpty_in.round().values, "payees per month": kyc.ncpty_out.round().values,
    "screened rank": kyc.screened_rank.values, "question": "should this client be registered as a TPPP?",
    "answer": "", "note": "", "reviewer": "", "reviewed_at": ""})
wb_arch = ARCH.assign(**{"look-alikes in this file": ARCH.archetype.map(lk.archetype.value_counts()).fillna(0).astype(int)})
readme = pd.DataFrame({"item": [
    "What this is", "Look-alikes", "Prospects", "KYC check", "Archetypes", "How reliable",
    "How to review", "Verdict values", "Window", "Confidentiality"], "text": [
    "Candidates found in the payment graph by comparing how money moves; a pattern worth checking, not a finding.",
    "PNC TM clients that are not registered as TPPPs but move money like the registered ones.",
    "Outside companies (not PNC clients) that move money like processors, with the TM clients who already pay them.",
    "Clients whose industry code or name says processor, not registered, and not on the look-alike list.",
    "Groups of registered TPPPs by behaviour; each look-alike shows the group it resembles.",
    f"Hidden registered TPPPs: {found(REC_S1, 'all registered TPPPs')} came back in the top 1% of {len(CS):,} clients.",
    "Fill verdict, note, reviewer and date; leave rows you did not check empty. Do not reorder or delete rows.",
    VERDICTS, f"{WINDOW_START} to {WINDOW_END}; amounts are monthly averages {WINDOW_START} to {STABLE_END}.",
    "Contains client names. Internal, TPPP / TM working group only; prospecting use pending the permissible-use review."]})
try:
    with pd.ExcelWriter(f_xl, engine="openpyxl") as xw:
        for df_, sh in [(readme, "README"), (wb_look, "Look-alikes"), (wb_pros, "Prospects"),
                        (wb_kyc, "KYC check"), (wb_arch, "Archetypes")]:
            df_.to_excel(xw, sheet_name=sh, index=False)
            ws = xw.sheets[sh]
            ws.freeze_panes = "B2" if sh != "README" else "A2"
            for k, c in enumerate(df_.columns, 1):
                q90 = int(df_[c].astype(str).str.len().quantile(0.9)) if len(df_) else 10
                ws.column_dimensions[ws.cell(1, k).column_letter].width = max(len(str(c)), min(60, q90)) + 2
            ws.auto_filter.ref = ws.dimensions
    xl_msg = f"written: {f_xl}"
except ImportError:
    xl_msg = "openpyxl not installed: workbook not written (pip install openpyxl)"
show("§11 Review workbook",
     ("Sheets", pd.DataFrame([{"sheet": "README", "rows": len(readme)}, {"sheet": "Look-alikes", "rows": len(wb_look)},
                               {"sheet": "Prospects", "rows": len(wb_pros)}, {"sheet": "KYC check", "rows": len(wb_kyc)},
                               {"sheet": "Archetypes", "rows": len(wb_arch)}])),
     note(xl_msg),
     note("Share only in the working group's access-controlled folder."))